In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

real = pd.read_csv('real.csv')
real['label'] = 0
fake = pd.read_csv('fake.csv')
fake['label'] = 1
df = pd.concat([real, fake])

df['tweet'] = df['tweet'].fillna('')  # Fill NaN values with empty strings
X = df['tweet']  # Adjust column name
y = df['label']

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)



In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=max_len
        )
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item


train_dataset = FakeNewsDataset(X_train, y_train, tokenizer)
val_dataset = FakeNewsDataset(X_val, y_val, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
max_len=64
val_loader = DataLoader(val_dataset, batch_size=16)


model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)


for epoch in range(3):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 0.10930695110114083
Epoch 2, Loss: 0.0397895465229919
Epoch 3, Loss: 0.01642362955135294


In [ ]:
import os
hugg_token=os.environ.get('huggingface_token')

In [ ]:
!pip install huggingface_hub transformers

In [ ]:
from huggingface_hub import login
login()

In [ ]:
model.save_pretrained("./football_fake_news_bert")
tokenizer.save_pretrained("./football_fake_news_bert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./football_fake_news_bert/tokenizer_config.json',
 './football_fake_news_bert/tokenizer.json')

In [ ]:
from huggingface_hub import create_repo

create_repo("football-fake-news-bert", private=False)

RepoUrl('https://huggingface.co/TSGN47/football-fake-news-bert', endpoint='https://huggingface.co', repo_type='model', repo_id='TSGN47/football-fake-news-bert')

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model.push_to_hub("football-fake-news-bert")
tokenizer.push_to_hub("football-fake-news-bert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...k55k4ti/model.safetensors:   0%|          | 14.2kB /  438MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/TSGN47/football-fake-news-bert/commit/59cadf89ded195bf93711b252f50f13f25b65b22', commit_message='Upload tokenizer', commit_description='', oid='59cadf89ded195bf93711b252f50f13f25b65b22', pr_url=None, repo_url=RepoUrl('https://huggingface.co/TSGN47/football-fake-news-bert', endpoint='https://huggingface.co', repo_type='model', repo_id='TSGN47/football-fake-news-bert'), pr_revision=None, pr_num=None)

In [ ]:
from sklearn.metrics import classification_report

model.eval()
preds = []
true_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        probs = torch.softmax(logits, dim=1)
        threshold = 0.4   # <-- change this value
        predictions = (probs[:, 1] > threshold).long()

        preds.extend(predictions.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

print(classification_report(true_labels, preds))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97      4374
           1       0.98      0.96      0.97      4000

    accuracy                           0.97      8374
   macro avg       0.97      0.97      0.97      8374
weighted avg       0.97      0.97      0.97      8374

